# 12 — Backtest v2: 2018 & 2022 World Cups

Backtests the eloratings.net formula (v2) against both 2018 and 2022 World Cups.
For each tournament:
- Elo is rebuilt on pre-tournament data only
- Poisson model is retrained on pre-tournament data
- Match outcomes are predicted and scored
- 10,000 Monte Carlo simulations run to get stage probabilities

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import poisson
from statsmodels.formula.api import glm
from statsmodels.genmod.families import Poisson
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

In [ ]:
df = pd.read_csv(r"C:\Users\aryan\OneDrive\Desktop\World Cup Predictor\results_clean.csv")
df['date'] = pd.to_datetime(df['date'])
print(df.shape)
print(f"Date range: {df['date'].min().date()} to {df['date'].max().date()}")

## Helper functions — eloratings.net formula

In [ ]:
K_VALUES = {
    'FIFA World Cup': 60,
    'UEFA Euro': 50, 'Copa América': 50, 'African Cup of Nations': 50,
    'AFC Asian Cup': 50, 'Gold Cup': 50, 'Oceania Nations Cup': 50,
    'Confederations Cup': 50, 'CONMEBOL-UEFA Cup of Champions': 50,
    'FIFA World Cup qualification': 40,
    'UEFA Euro qualification': 40, 'African Cup of Nations qualification': 40,
    'AFC Asian Cup qualification': 40, 'Copa América qualification': 40,
    'Gold Cup qualification': 40, 'CONCACAF Nations League qualification': 40,
    'Oceania Nations Cup qualification': 40,
    'UEFA Nations League': 40, 'CONCACAF Nations League': 40,
    'AFC Challenge Cup': 40, 'AFC Challenge Cup qualification': 40,
    'Friendly': 20,
}
DEFAULT_K = 30

def get_k(tournament):
    return K_VALUES.get(tournament, DEFAULT_K)

def goal_diff_multiplier(goal_diff):
    n = abs(goal_diff)
    if n <= 1:  return 1.0
    elif n == 2: return 1.5
    elif n == 3: return 1.75
    else:        return 1.75 + (n - 3) / 8

def expected_score(ra, rb):
    return 1 / (1 + 10 ** ((rb - ra) / 400))

def match_probs(home_xg, away_xg, max_goals=10):
    home_win, draw, away_win = 0, 0, 0
    for i in range(max_goals):
        for j in range(max_goals):
            p = poisson.pmf(i, home_xg) * poisson.pmf(j, away_xg)
            if i > j: home_win += p
            elif i == j: draw += p
            else: away_win += p
    return home_win, draw, away_win

print("Helper functions ready!")

## Function: Build Elo from scratch up to a cutoff date

In [ ]:
def build_elo_v2(train_df):
    """Build Elo ratings using eloratings.net formula on training data."""
    elo = {}

    def get_e(team):
        if team not in elo: elo[team] = 1500
        return elo[team]

    for _, row in train_df.sort_values('date').iterrows():
        ht, at = row['home_team'], row['away_team']
        hs, as_ = row['home_score'], row['away_score']
        neutral = row['neutral']
        tournament = row['tournament']

        he, ae = get_e(ht), get_e(at)
        we_home = expected_score(he + (0 if neutral else 100), ae)
        we_away = 1 - we_home

        if hs > as_:   w_home, w_away = 1, 0
        elif hs < as_: w_home, w_away = 0, 1
        else:          w_home, w_away = 0.5, 0.5

        k = get_k(tournament) * goal_diff_multiplier(abs(hs - as_))

        elo[ht] = he + k * (w_home - we_home)
        elo[at] = ae + k * (w_away - we_away)

    return elo

print("build_elo_v2 ready!")

## Function: Train Poisson model on pre-tournament data

In [ ]:
def train_poisson(train_df, elo_ratings):
    """Merge Elo into training data and fit Poisson GLMs."""
    elo_df = pd.DataFrame(list(elo_ratings.items()), columns=['team', 'elo'])

    features = train_df.copy()
    features = features.merge(elo_df, left_on='home_team', right_on='team', how='left').rename(columns={'elo': 'home_elo'}).drop(columns='team')
    features = features.merge(elo_df, left_on='away_team', right_on='team', how='left').rename(columns={'elo': 'away_elo'}).drop(columns='team')
    features['elo_diff'] = features['home_elo'] - features['away_elo']
    features['neutral'] = features['neutral'].astype(int)
    features['fifa_rank_diff'] = 0
    features = features.dropna(subset=['home_elo', 'away_elo', 'home_score', 'away_score'])

    home_m = glm('home_score ~ home_elo + away_elo + elo_diff + neutral + tournament_weight',
                 data=features, family=Poisson()).fit()
    away_m = glm('away_score ~ home_elo + away_elo + elo_diff + neutral + tournament_weight',
                 data=features, family=Poisson()).fit()

    print(f"  Home R²: {home_m.pseudo_rsquared():.3f} | Away R²: {away_m.pseudo_rsquared():.3f}")
    return home_m, away_m

print("train_poisson ready!")

## Function: Predict & score match outcomes

In [ ]:
def predict_and_score(test_df, elo_ratings, home_m, away_m):
    predictions = []

    for _, row in test_df.iterrows():
        he = elo_ratings.get(row['home_team'], 1500)
        ae = elo_ratings.get(row['away_team'], 1500)

        match = pd.DataFrame([{
            'home_elo': he, 'away_elo': ae,
            'elo_diff': he - ae,
            'neutral': 1,
            'tournament_weight': 5.0,
        }])

        home_xg = home_m.predict(match)[0]
        away_xg = away_m.predict(match)[0]
        hw, d, aw = match_probs(home_xg, away_xg)

        actual = 'H' if row['home_score'] > row['away_score'] else ('A' if row['home_score'] < row['away_score'] else 'D')
        predicted = 'H' if hw > d and hw > aw else ('D' if d > aw else 'A')

        predictions.append({
            'home_team': row['home_team'], 'away_team': row['away_team'],
            'home_score': row['home_score'], 'away_score': row['away_score'],
            'home_xg': round(home_xg, 2), 'away_xg': round(away_xg, 2),
            'p_home_win': round(hw, 3), 'p_draw': round(d, 3), 'p_away_win': round(aw, 3),
            'actual': actual, 'predicted': predicted
        })

    pred_df = pd.DataFrame(predictions)

    # Metrics
    accuracy = (pred_df['actual'] == pred_df['predicted']).mean()

    def brier(row):
        ap = {'H': 0, 'D': 0, 'A': 0}; ap[row['actual']] = 1
        return (row['p_home_win']-ap['H'])**2 + (row['p_draw']-ap['D'])**2 + (row['p_away_win']-ap['A'])**2

    def logloss(row):
        ap = {'H': 0, 'D': 0, 'A': 0}; ap[row['actual']] = 1; eps = 1e-10
        return -(ap['H']*np.log(row['p_home_win']+eps) + ap['D']*np.log(row['p_draw']+eps) + ap['A']*np.log(row['p_away_win']+eps))

    pred_df['brier'] = pred_df.apply(brier, axis=1)
    pred_df['log_loss'] = pred_df.apply(logloss, axis=1)

    return pred_df, accuracy, pred_df['brier'].mean(), pred_df['log_loss'].mean()

print("predict_and_score ready!")

## 2022 World Cup Backtest

In [ ]:
WC2022_START = '2022-11-20'

train_2022 = df[df['date'] < WC2022_START].copy()
test_2022  = df[(df['date'] >= WC2022_START) & (df['tournament'] == 'FIFA World Cup')].copy()

print(f"Training matches: {len(train_2022)}")
print(f"2022 WC matches:  {len(test_2022)}")

print("\nBuilding Elo v2 on pre-2022 data...")
elo_2022 = build_elo_v2(train_2022)
print(f"Teams rated: {len(elo_2022)}")

# Top 10 just before 2022 WC
elo_2022_df = pd.DataFrame(list(elo_2022.items()), columns=['team', 'elo']).sort_values('elo', ascending=False)
print("\nTop 10 just before 2022 WC:")
print(elo_2022_df.head(10).to_string(index=False))

In [ ]:
print("Training Poisson model on pre-2022 data...")
home_m_2022, away_m_2022 = train_poisson(train_2022, elo_2022)

In [ ]:
pred_2022, acc_2022, brier_2022, ll_2022 = predict_and_score(test_2022, elo_2022, home_m_2022, away_m_2022)

print("=== 2022 WC BACKTEST (v2 — eloratings formula) ===")
print(f"Matches:         {len(pred_2022)}")
print(f"Accuracy:        {acc_2022*100:.1f}%  (random = 33.3%)")
print(f"Brier Score:     {brier_2022:.4f}  (random = 0.667)")
print(f"Log Loss:        {ll_2022:.4f}  (random = 1.099)")
print(f"Draws predicted: {(pred_2022['predicted']=='D').sum()} out of {(pred_2022['actual']=='D').sum()} actual draws")
print()
print("Prediction breakdown:")
print(pred_2022.groupby(['actual', 'predicted']).size().unstack(fill_value=0))

In [ ]:
print("Biggest upsets we got wrong:")
wrong = pred_2022[pred_2022['actual'] != pred_2022['predicted']].sort_values('brier', ascending=False)
print(wrong[['home_team', 'away_team', 'home_score', 'away_score',
             'p_home_win', 'p_draw', 'p_away_win', 'actual', 'predicted']].head(10).to_string(index=False))

## 2022 Monte Carlo — 10,000 simulations

In [ ]:
wc2022_groups = {
    'A': ['Qatar', 'Ecuador', 'Senegal', 'Netherlands'],
    'B': ['England', 'Iran', 'United States', 'Wales'],
    'C': ['Argentina', 'Saudi Arabia', 'Mexico', 'Poland'],
    'D': ['France', 'Australia', 'Denmark', 'Tunisia'],
    'E': ['Spain', 'Costa Rica', 'Germany', 'Japan'],
    'F': ['Belgium', 'Canada', 'Morocco', 'Croatia'],
    'G': ['Brazil', 'Serbia', 'Switzerland', 'Cameroon'],
    'H': ['Portugal', 'Ghana', 'Uruguay', 'South Korea'],
}

wc2022_fixtures = []
for group, teams in wc2022_groups.items():
    for i in range(len(teams)):
        for j in range(i+1, len(teams)):
            wc2022_fixtures.append({'home_team': teams[i], 'away_team': teams[j], 'group': group})
fixtures_2022 = pd.DataFrame(wc2022_fixtures)
print(f"Group stage fixtures: {len(fixtures_2022)}")

def predict_bt(ht, at, hm, am, elo):
    he = elo.get(ht, 1500); ae = elo.get(at, 1500)
    match = pd.DataFrame([{'home_elo': he, 'away_elo': ae, 'elo_diff': he-ae, 'neutral': 1, 'tournament_weight': 5.0}])
    return hm.predict(match)[0], am.predict(match)[0]

def sim_match(ht, at, hm, am, elo):
    hxg, axg = predict_bt(ht, at, hm, am, elo)
    return np.random.poisson(hxg), np.random.poisson(axg)

def sim_knockout(ht, at, hm, am, elo):
    hxg, axg = predict_bt(ht, at, hm, am, elo)
    hg, ag = np.random.poisson(hxg), np.random.poisson(axg)
    if hg == ag:
        et_h, et_a = np.random.poisson(hxg*0.33), np.random.poisson(axg*0.33)
        hg += et_h; ag += et_a
    if hg == ag:
        he = elo.get(ht, 1500); ae = elo.get(at, 1500)
        pen_prob = np.clip(0.5 + (he-ae)*0.0001, 0.4, 0.6)
        hg += 1 if np.random.random() < pen_prob else 0
        ag += 0 if hg > ag else 1
    return ht if hg > ag else at

def sim_group_2022(group_name, group_teams, hm, am, elo):
    table = {t: {'W':0,'D':0,'L':0,'GF':0,'GA':0,'Pts':0} for t in group_teams}
    gf = fixtures_2022[fixtures_2022['group']==group_name]
    for _, row in gf.iterrows():
        h, a = row['home_team'], row['away_team']
        hg, ag = sim_match(h, a, hm, am, elo)
        table[h]['GF']+=hg; table[h]['GA']+=ag
        table[a]['GF']+=ag; table[a]['GA']+=hg
        if hg>ag: table[h]['W']+=1; table[h]['Pts']+=3; table[a]['L']+=1
        elif hg<ag: table[a]['W']+=1; table[a]['Pts']+=3; table[h]['L']+=1
        else: table[h]['D']+=1; table[a]['D']+=1; table[h]['Pts']+=1; table[a]['Pts']+=1
    tdf = pd.DataFrame(table).T
    tdf['GD'] = tdf['GF']-tdf['GA']
    return tdf.sort_values(['Pts','GD','GF'], ascending=False).reset_index()

def sim_wc2022(hm, am, elo):
    winners, runners = [], []
    for g, teams in wc2022_groups.items():
        t = sim_group_2022(g, teams, hm, am, elo)
        winners.append(t.iloc[0]['index'])
        runners.append(t.iloc[1]['index'])
    r16_pairs = [(winners[i], runners[j]) for i,j in [(0,1),(1,0),(2,3),(3,2),(4,5),(5,4),(6,7),(7,6)]]
    r16 = [sim_knockout(h,a,hm,am,elo) for h,a in r16_pairs]
    qf  = [sim_knockout(r16[i],r16[i+1],hm,am,elo) for i in range(0,8,2)]
    sf  = [sim_knockout(qf[i],qf[i+1],hm,am,elo) for i in range(0,4,2)]
    win = sim_knockout(sf[0],sf[1],hm,am,elo)
    return {'r16': r16, 'qf': qf, 'sf': sf, 'winner': win}

print("Simulation functions ready!")

In [ ]:
N = 10000
counts_2022 = defaultdict(lambda: defaultdict(int))

print(f"Running {N:,} simulations of 2022 World Cup...")
for sim in range(N):
    if sim % 1000 == 0: print(f"  {sim}/{N}...")
    r = sim_wc2022(home_m_2022, away_m_2022, elo_2022)
    for t in r['r16']: counts_2022[t]['R16'] += 1
    for t in r['qf']:  counts_2022[t]['QF']  += 1
    for t in r['sf']:  counts_2022[t]['SF']  += 1
    for t in r['sf']:  counts_2022[t]['Final'] += 1
    counts_2022[r['winner']]['Winner'] += 1

print("Done!")

In [ ]:
all_2022 = [t for g in wc2022_groups.values() for t in g]
res_2022 = []
for team in all_2022:
    res_2022.append({
        'Team': team,
        'R16 %':   round(counts_2022[team]['R16']   / N * 100, 1),
        'QF %':    round(counts_2022[team]['QF']    / N * 100, 1),
        'SF %':    round(counts_2022[team]['SF']    / N * 100, 1),
        'Final %': round(counts_2022[team]['Final'] / N * 100, 1),
        'Winner %':round(counts_2022[team]['Winner']/ N * 100, 1),
    })

actual_2022 = {
    'Argentina': 'WINNER', 'France': 'Final',
    'Croatia': 'SF', 'Morocco': 'SF',
    'Netherlands': 'QF', 'England': 'QF', 'Brazil': 'QF', 'Portugal': 'QF',
}

res_2022_df = pd.DataFrame(res_2022).sort_values('Winner %', ascending=False).reset_index(drop=True)
res_2022_df.index += 1
res_2022_df['Actual'] = res_2022_df['Team'].map(actual_2022).fillna('R16 or less')

print("2022 World Cup Simulation Results (10,000 runs) — v2")
print(res_2022_df.to_string())

In [ ]:
# Model scoring vs actual deep runners
deep_runs_2022 = {
    'Argentina': 6, 'France': 5, 'Croatia': 4, 'Morocco': 4,
    'Netherlands': 3, 'England': 3, 'Brazil': 3, 'Portugal': 3,
}

print("=== MODEL SCORING vs ACTUAL 2022 RESULTS ===\n")
print(f"{'Team':<15} {'Predicted Rank':<16} {'Actual Stage':<12} {'Verdict'}")
print("-" * 55)
for team, _ in sorted(deep_runs_2022.items(), key=lambda x: -x[1]):
    rank = res_2022_df[res_2022_df['Team'] == team].index[0]
    verdict = '✅' if rank <= 10 else '⚠️' if rank <= 16 else '❌'
    stage = res_2022_df[res_2022_df['Team'] == team]['Actual'].values[0]
    print(f"{team:<15} #{rank:<15} {stage:<12} {verdict}")

## 2018 World Cup Backtest

In [ ]:
WC2018_START = '2018-06-14'

train_2018 = df[df['date'] < WC2018_START].copy()
test_2018  = df[(df['date'] >= WC2018_START) & (df['tournament'] == 'FIFA World Cup') & (df['date'] < '2018-07-16')].copy()

print(f"Training matches: {len(train_2018)}")
print(f"2018 WC matches:  {len(test_2018)}")

print("\nBuilding Elo v2 on pre-2018 data...")
elo_2018 = build_elo_v2(train_2018)

elo_2018_df = pd.DataFrame(list(elo_2018.items()), columns=['team', 'elo']).sort_values('elo', ascending=False)
print("\nTop 10 just before 2018 WC:")
print(elo_2018_df.head(10).to_string(index=False))

In [ ]:
print("Training Poisson model on pre-2018 data...")
home_m_2018, away_m_2018 = train_poisson(train_2018, elo_2018)

In [ ]:
pred_2018, acc_2018, brier_2018, ll_2018 = predict_and_score(test_2018, elo_2018, home_m_2018, away_m_2018)

print("=== 2018 WC BACKTEST (v2 — eloratings formula) ===")
print(f"Matches:         {len(pred_2018)}")
print(f"Accuracy:        {acc_2018*100:.1f}%  (random = 33.3%)")
print(f"Brier Score:     {brier_2018:.4f}  (random = 0.667)")
print(f"Log Loss:        {ll_2018:.4f}  (random = 1.099)")
print(f"Draws predicted: {(pred_2018['predicted']=='D').sum()} out of {(pred_2018['actual']=='D').sum()} actual draws")
print()
print("Prediction breakdown:")
print(pred_2018.groupby(['actual', 'predicted']).size().unstack(fill_value=0))

## 2018 Monte Carlo — 10,000 simulations

In [ ]:
wc2018_groups = {
    'A': ['Russia', 'Saudi Arabia', 'Egypt', 'Uruguay'],
    'B': ['Portugal', 'Spain', 'Morocco', 'Iran'],
    'C': ['France', 'Australia', 'Peru', 'Denmark'],
    'D': ['Argentina', 'Iceland', 'Croatia', 'Nigeria'],
    'E': ['Brazil', 'Switzerland', 'Costa Rica', 'Serbia'],
    'F': ['Germany', 'Mexico', 'Sweden', 'South Korea'],
    'G': ['Belgium', 'Panama', 'Tunisia', 'England'],
    'H': ['Poland', 'Senegal', 'Colombia', 'Japan'],
}

wc2018_fixtures = []
for group, teams in wc2018_groups.items():
    for i in range(len(teams)):
        for j in range(i+1, len(teams)):
            wc2018_fixtures.append({'home_team': teams[i], 'away_team': teams[j], 'group': group})
fixtures_2018 = pd.DataFrame(wc2018_fixtures)
print(f"Group stage fixtures: {len(fixtures_2018)}")

def sim_group_2018(group_name, group_teams, hm, am, elo):
    table = {t: {'W':0,'D':0,'L':0,'GF':0,'GA':0,'Pts':0} for t in group_teams}
    gf = fixtures_2018[fixtures_2018['group']==group_name]
    for _, row in gf.iterrows():
        h, a = row['home_team'], row['away_team']
        hg, ag = sim_match(h, a, hm, am, elo)
        table[h]['GF']+=hg; table[h]['GA']+=ag
        table[a]['GF']+=ag; table[a]['GA']+=hg
        if hg>ag: table[h]['W']+=1; table[h]['Pts']+=3; table[a]['L']+=1
        elif hg<ag: table[a]['W']+=1; table[a]['Pts']+=3; table[h]['L']+=1
        else: table[h]['D']+=1; table[a]['D']+=1; table[h]['Pts']+=1; table[a]['Pts']+=1
    tdf = pd.DataFrame(table).T
    tdf['GD'] = tdf['GF']-tdf['GA']
    return tdf.sort_values(['Pts','GD','GF'], ascending=False).reset_index()

def sim_wc2018(hm, am, elo):
    winners, runners = [], []
    for g, teams in wc2018_groups.items():
        t = sim_group_2018(g, teams, hm, am, elo)
        winners.append(t.iloc[0]['index'])
        runners.append(t.iloc[1]['index'])
    r16_pairs = [(winners[i], runners[j]) for i,j in [(0,1),(1,0),(2,3),(3,2),(4,5),(5,4),(6,7),(7,6)]]
    r16 = [sim_knockout(h,a,hm,am,elo) for h,a in r16_pairs]
    qf  = [sim_knockout(r16[i],r16[i+1],hm,am,elo) for i in range(0,8,2)]
    sf  = [sim_knockout(qf[i],qf[i+1],hm,am,elo) for i in range(0,4,2)]
    win = sim_knockout(sf[0],sf[1],hm,am,elo)
    return {'r16': r16, 'qf': qf, 'sf': sf, 'winner': win}

print("2018 simulation functions ready!")

In [ ]:
N = 10000
counts_2018 = defaultdict(lambda: defaultdict(int))

print(f"Running {N:,} simulations of 2018 World Cup...")
for sim in range(N):
    if sim % 1000 == 0: print(f"  {sim}/{N}...")
    r = sim_wc2018(home_m_2018, away_m_2018, elo_2018)
    for t in r['r16']: counts_2018[t]['R16'] += 1
    for t in r['qf']:  counts_2018[t]['QF']  += 1
    for t in r['sf']:  counts_2018[t]['SF']  += 1
    for t in r['sf']:  counts_2018[t]['Final'] += 1
    counts_2018[r['winner']]['Winner'] += 1

print("Done!")

In [ ]:
all_2018 = [t for g in wc2018_groups.values() for t in g]
res_2018 = []
for team in all_2018:
    res_2018.append({
        'Team': team,
        'R16 %':   round(counts_2018[team]['R16']   / N * 100, 1),
        'QF %':    round(counts_2018[team]['QF']    / N * 100, 1),
        'SF %':    round(counts_2018[team]['SF']    / N * 100, 1),
        'Final %': round(counts_2018[team]['Final'] / N * 100, 1),
        'Winner %':round(counts_2018[team]['Winner']/ N * 100, 1),
    })

actual_2018 = {
    'France': 'WINNER', 'Croatia': 'Final',
    'Belgium': 'SF', 'England': 'SF',
    'Brazil': 'QF', 'Uruguay': 'QF', 'Russia': 'QF', 'Sweden': 'QF',
}

res_2018_df = pd.DataFrame(res_2018).sort_values('Winner %', ascending=False).reset_index(drop=True)
res_2018_df.index += 1
res_2018_df['Actual'] = res_2018_df['Team'].map(actual_2018).fillna('R16 or less')

print("2018 World Cup Simulation Results (10,000 runs) — v2")
print(res_2018_df.to_string())

In [ ]:
deep_runs_2018 = {
    'France': 6, 'Croatia': 5, 'Belgium': 4, 'England': 4,
    'Brazil': 3, 'Uruguay': 3, 'Russia': 3, 'Sweden': 3,
}

print("=== MODEL SCORING vs ACTUAL 2018 RESULTS ===\n")
print(f"{'Team':<15} {'Predicted Rank':<16} {'Actual Stage':<12} {'Verdict'}")
print("-" * 55)
for team, _ in sorted(deep_runs_2018.items(), key=lambda x: -x[1]):
    rank = res_2018_df[res_2018_df['Team'] == team].index[0]
    verdict = '✅' if rank <= 10 else '⚠️' if rank <= 16 else '❌'
    stage = res_2018_df[res_2018_df['Team'] == team]['Actual'].values[0]
    print(f"{team:<15} #{rank:<15} {stage:<12} {verdict}")

## Summary: v1 vs v2 across both tournaments

In [ ]:
print("=" * 55)
print("BACKTEST SUMMARY — v2 (eloratings formula)")
print("=" * 55)
print(f"{'Metric':<20} {'2018':>10} {'2022':>10}")
print("-" * 42)
print(f"{'Accuracy':<20} {acc_2018*100:>9.1f}% {acc_2022*100:>9.1f}%")
print(f"{'Brier Score':<20} {brier_2018:>10.4f} {brier_2022:>10.4f}")
print(f"{'Log Loss':<20} {ll_2018:>10.4f} {ll_2022:>10.4f}")
print(f"{'Draws predicted':<20} {(pred_2018['predicted']=='D').sum():>10} {(pred_2022['predicted']=='D').sum():>10}")
print(f"{'Actual draws':<20} {(pred_2018['actual']=='D').sum():>10} {(pred_2022['actual']=='D').sum():>10}")
print("=" * 55)